In [1]:
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from match.achilles.model import Achilles
from instant_match.components import (
    build_features
)

from instant_match.loader.startup import get_vectorizer_path

local_vectorizer_path = get_vectorizer_path("PH", "5_3")
project_id = "dh-global-sales-data-dev"

2025-02-28 17:11:01,598 - match.achilles - WARNING - Usage of /Users/linda.mayorga/git-repos/gsd-match-vertex/match_core/match/achilles/feature.py has been deprecated.Use achilles/feature_pyspark.py module instead.


In [2]:
query = f"""
SELECT *
FROM `dh-global-sales-data-dev.achilles.training_candidates_new`
where country_iso = 'PH'
and model_version = '5'
and left_row_id not like 'ext_row_%'
"""

df_training_candidates_new_ph_v5 = pd.read_gbq(query=query, project_id = project_id, progress_bar_type = 'tqdm')

/opt/homebrew/anaconda3/envs/embedding_experimentation/lib/python3.9/site-packages/google/cloud/bigquery/table.py:2309: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/homebrew/anaconda3/envs/embedding_experimentation/lib/python3.9/site-packages/google/cloud/bigquery/table.py:2323: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/homebrew/anaconda3/envs/embedding_experimentation/lib/python3.9/site-packages/google/cloud/bigquery/table.py:2337: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)


Downloading: 100%|█████████████████████████████████████████████████████████████████████████████████████|


In [3]:
df_training_candidates_new_ph_v5.head()

,country_iso,left_row_id,left_name,left_street,left_lat,left_lng,left_phone_number,left_street_stop,left_name_stop,left_name_stop_phonetic,...,left_name_local_transliterated,right_name_local_transliterated,right_name_legal,model_id,col_type,left_registration_number,right_registration_number,data_category_type,left_area,right_area
0,PH,FP_PH----anakin---grab---PHGFSTI0000000k,contis bakeshop and restaurant,1c 17 serendra taguig city taguig 1630,14.549699,121.054102,None,1 17 serendra 1630,contis,KNTS,...,contis,contis uptown place mall,None,PH,train,None,None,geo,None,None
1,PH,FP_PH----anakin---grab---PHGFSTI0000000k,contis bakeshop and restaurant,1c 17 serendra taguig city taguig 1630,14.549699,121.054102,None,1 17 serendra 1630,contis,KNTS,...,contis,contis uptown place mall,None,PH,train,None,None,geo,None,None
2,PH,FP_PH----anakin---grab---PHGFSTI000000g4,nyfd new york fries dips,3rd floor up town center 1108 katipunan avenue...,14.649632,121.074498,None,3 up 1108 katipunan diliman 1101,nyfd york fries dips,NFT YRK FRS TPS,...,nyfd york fries dips,new york fries and dips up town center,None,PH,train,None,None,geo,None,None
3,PH,FP_PH----anakin---grab---PHGFSTI000000yd,pho hoa vietnamese noodle house,eton centris centris walk quezon avenue corner...,14.641830,121.039470,None,eton centris walk edsa pinyahan 1100,pho hoa vietnamese noodle,F H FTNMS NTL,...,pho hoa vietnamese noodle,pho hoa centris,None,PH,train,None,None,geo,None,None
4,PH,FP_PH----anakin---grab---PHGFSTI000001sr,famous belgian waffles,4th floor sm north edsa the block epifanio de ...,14.621585,121.019419,None,4 edsa epifanio de los santos 1105,famous belgian waffles,FMS BLJN WFLS,...,famous belgian waffles,famous belgian waffles sm city sta mesa,None,PH,train,None,None,geo,None,None


In [4]:
candidates = df_training_candidates_new_ph_v5.drop(columns=["model_id", "model_version", "col_type"])

In [5]:
features = [
    "country_iso",
    "left_row_id",
    "right_row_id",
    "levenshtein_name_stop_phonetic",
    "levenshtein_street_stop_phonetic",
    "wratio_name",
    "jaro_winkler_name",
    "jaro_winkler_name_stop",
    "jaro_winkler_name_local",
    "jaro_winkler_name_local_transliterated",
    "jaro_winkler_name_reversed",
    "jw_name_local_ascii_only",
    "jw_name_local_nonascii_only",
    "tokenset_name_local",
    "tokenset_name_legal",
    "tokenset_name_local_nonascii_only",
    "tokenset_name_local_ascii_only",
    "tokenset_name_local_transliterated",
    "tokenset_name",
    "tokenset_name_stop",
    "tokenset_street_stop",
    "wratio_name_stop",
    "wratio_street_stop",
    "cosine_name_stop",
    "haversine",
    "same_phone",
]

In [6]:
feature_pipeline = Achilles(
        model_id="PH",
        model_version="5",
        local_vectoriser_path=local_vectorizer_path,
        feature_list=features,
    )

In [7]:
features_df = build_features(candidates, feature_pipeline)

2025-02-28 17:13:46,660 - match.achilles - WARNING - Usage of /Users/linda.mayorga/git-repos/gsd-match-vertex/match_core/match/achilles/feature.py has been deprecated.Use achilles/feature_pyspark.py module instead.
2025-02-28 17:13:46,689 - match.achilles - WARNING - Usage of /Users/linda.mayorga/git-repos/gsd-match-vertex/match_core/match/achilles/feature.py has been deprecated.Use achilles/feature_pyspark.py module instead.
2025-02-28 17:13:46,698 - match.achilles - DEBUG - <class 'match.achilles.feature.RowDistanceMaker'>
2025-02-28 17:13:46,711 - match.achilles - WARNING - Usage of /Users/linda.mayorga/git-repos/gsd-match-vertex/match_core/match/achilles/feature.py has been deprecated.Use achilles/feature_pyspark.py module instead.
2025-02-28 17:13:46,712 - match.achilles - WARNING - Usage of /Users/linda.mayorga/git-repos/gsd-match-vertex/match_core/match/achilles/feature.py has been deprecated.Use achilles/feature_pyspark.py module instead.
2025-02-28 17:13:46,729 - match.achille

In [8]:
features_df.head()

,country_iso,left_row_id,right_row_id,levenshtein_name_stop_phonetic,levenshtein_street_stop_phonetic,wratio_name,jaro_winkler_name,jaro_winkler_name_stop,jaro_winkler_name_local,jaro_winkler_name_local_transliterated,...,tokenset_name_local_ascii_only,tokenset_name_local_transliterated,tokenset_name,tokenset_name_stop,tokenset_street_stop,wratio_name_stop,wratio_street_stop,cosine_name_stop,haversine,same_phone
0,PH,FP_PH----anakin---grab---PHGFSTI0000000k,FP_PH----salesforce---salesforce---45PAGX,1.000000,NaN,0.855,0.848039,1.000000,0.850000,0.850000,...,0.444444,1.000000,0.520000,1.000000,0.1,1.000,0.3,1.000000,1.010111,0
1,PH,FP_PH----anakin---grab---PHGFSTI0000000k,FP_PH----salesforce---salesforce---45PAGX,1.000000,NaN,0.855,0.848039,1.000000,0.850000,0.850000,...,0.444444,1.000000,0.520000,1.000000,0.1,1.000,0.3,1.000000,1.010111,0
2,PH,FP_PH----anakin---grab---PHGFSTI000000g4,FP_PH----salesforce---salesforce---45PTXI,0.733333,0.357143,0.855,0.753230,0.805556,0.698635,0.698635,...,0.583333,0.857143,0.883721,1.000000,1.0,0.950,0.9,0.787097,0.140724,0
3,PH,FP_PH----anakin---grab---PHGFSTI000000yd,FP_PH----salesforce---salesforce---45PP0Z,0.461538,NaN,0.855,0.841820,0.861949,0.861949,0.861949,...,0.514286,0.636364,0.636364,0.636364,NaN,0.855,NaN,0.454342,0.091627,0
4,PH,FP_PH----anakin---grab---PHGFSTI000001sr,FP_PH----salesforce---salesforce---HK07TL,1.000000,NaN,0.900,0.912821,1.000000,0.912821,0.912821,...,0.754717,1.000000,1.000000,1.000000,NaN,1.000,NaN,1.000000,1.872155,0


In [9]:
def generate_vendor_document(name, name_local, street):
    return f"""name: {name}, name_local: {name_local}, street: {street}"""

In [10]:
def generate_sentence_transformer_embeddings(documents):
    batch_size = 128
    embeddings = []
    model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
    for start_idx in tqdm(range(0, len(documents), batch_size)):
        end_idx = start_idx + batch_size
        batch = documents[start_idx:end_idx]
        embeddings_batch = model.encode(batch)
        embeddings.extend(embeddings_batch)
    return embeddings

In [21]:
def create_semantic_features(df):
    #Generate document
    print("Generate document")
    df["left_document"] = df.apply(lambda x: generate_vendor_document(x.left_name, x.left_name_local, x.left_street), axis=1)
    df["right_document"] = df.apply(lambda x: generate_vendor_document(x.right_name, x.right_name_local, x.right_street), axis=1)

    
    #Transform to list
    print("Transform to list")
    left_documents = df["left_document"].values.tolist()
    right_documents = df["right_document"].values.tolist()
    left_names = df["left_name"].values.tolist()
    right_names = df["right_name"].values.tolist()
    left_names_local = df["left_name_local"].values.tolist()
    right_names_local = df["right_name_local"].values.tolist()

    #Generate embeddings
    print("Generate embeddings")
    left_document_embeddings = generate_sentence_transformer_embeddings(left_documents)
    right_document_embeddings = generate_sentence_transformer_embeddings(right_documents)
    left_name_embeddings = generate_sentence_transformer_embeddings(left_names)
    right_name_embeddings = generate_sentence_transformer_embeddings(right_names)
    left_name_local_embeddings = generate_sentence_transformer_embeddings(left_names_local)
    right_name_local_embeddings = generate_sentence_transformer_embeddings(right_names_local)

    #Compute similarity
    print("Compute similarity")
    doc_sem_similarity = [cosine_similarity([a], [b])[0][0] for a, b in zip(left_document_embeddings, right_document_embeddings)]
    name_sem_similarity = [cosine_similarity([a], [b])[0][0] for a, b in zip(left_name_embeddings, right_name_embeddings)]
    name_local_sem_similarity = [cosine_similarity([a], [b])[0][0] for a, b in zip(left_name_local_embeddings, right_name_local_embeddings)]

    #Store results
    print("Store results")
    df["doc_sem_similarity"] = doc_sem_similarity
    df["name_sem_similarity"] = name_sem_similarity
    df["name_local_sem_similarity"] = name_local_sem_similarity
    
    return df

In [22]:
candidates_w_semantic_features = create_semantic_features(candidates)

Generate document
Transform to list
Generate embeddings


100%|███████████████████████████████████████████████████████████████████| 34/34 [00:02<00:00, 15.36it/s]


Compute similarity
Store results


In [23]:
candidates_w_semantic_features.head()

,country_iso,left_row_id,left_name,left_street,left_lat,left_lng,left_phone_number,left_street_stop,left_name_stop,left_name_stop_phonetic,...,left_registration_number,right_registration_number,data_category_type,left_area,right_area,left_document,right_document,doc_sem_similarity,name_sem_similarity,name_local_sem_similarity
0,PH,FP_PH----anakin---grab---PHGFSTI0000000k,contis bakeshop and restaurant,1c 17 serendra taguig city taguig 1630,14.549699,121.054102,None,1 17 serendra 1630,contis,KNTS,...,None,None,geo,None,None,"name: contis bakeshop and restaurant, name_loc...","name: contis uptown parade, name_local: contis...",0.695060,0.333356,0.460829
1,PH,FP_PH----anakin---grab---PHGFSTI0000000k,contis bakeshop and restaurant,1c 17 serendra taguig city taguig 1630,14.549699,121.054102,None,1 17 serendra 1630,contis,KNTS,...,None,None,geo,None,None,"name: contis bakeshop and restaurant, name_loc...","name: contis uptown parade, name_local: contis...",0.695060,0.333356,0.460829
2,PH,FP_PH----anakin---grab---PHGFSTI000000g4,nyfd new york fries dips,3rd floor up town center 1108 katipunan avenue...,14.649632,121.074498,None,3 up 1108 katipunan diliman 1101,nyfd york fries dips,NFT YRK FRS TPS,...,None,None,geo,None,None,"name: nyfd new york fries dips, name_local: ny...","name: new york fries and dips up town center, ...",0.896492,0.829485,0.799608
3,PH,FP_PH----anakin---grab---PHGFSTI000000yd,pho hoa vietnamese noodle house,eton centris centris walk quezon avenue corner...,14.641830,121.039470,None,eton centris walk edsa pinyahan 1100,pho hoa vietnamese noodle,F H FTNMS NTL,...,None,None,geo,None,None,"name: pho hoa vietnamese noodle house, name_lo...","name: pho hoa centris, name_local: pho hoa cen...",0.727053,0.516631,0.544826
4,PH,FP_PH----anakin---grab---PHGFSTI000001sr,famous belgian waffles,4th floor sm north edsa the block epifanio de ...,14.621585,121.019419,None,4 edsa epifanio de los santos 1105,famous belgian waffles,FMS BLJN WFLS,...,None,None,geo,None,None,"name: famous belgian waffles, name_local: famo...","name: famous belgian waffles sm city sta mesa,...",0.880619,0.853448,0.853448


In [24]:
features_df_w_semantic_sims = candidates_w_semantic_features.drop(columns=["haversine"]).merge(features_df, how="left", left_on=["country_iso", "left_row_id", "right_row_id"], right_on=["country_iso", "left_row_id", "right_row_id"])

In [25]:
features_df_w_semantic_sims.columns

Index(['country_iso', 'left_row_id', 'left_name', 'left_street', 'left_lat',
       'left_lng', 'left_phone_number', 'left_street_stop', 'left_name_stop',
       'left_name_stop_phonetic', 'left_street_stop_phonetic', 'right_row_id',
       'right_name', 'right_street', 'right_lat', 'right_lng',
       'right_phone_number', 'right_street_stop', 'right_name_stop',
       'right_name_stop_phonetic', 'right_street_stop_phonetic', 'label',
       'left_name_local', 'right_name_local', 'left_name_local_transliterated',
       'right_name_local_transliterated', 'right_name_legal',
       'left_registration_number', 'right_registration_number',
       'data_category_type', 'left_area', 'right_area', 'left_document',
       'right_document', 'doc_sem_similarity', 'name_sem_similarity',
       'name_local_sem_similarity', 'levenshtein_name_stop_phonetic',
       'levenshtein_street_stop_phonetic', 'wratio_name', 'jaro_winkler_name',
       'jaro_winkler_name_stop', 'jaro_winkler_name_local',
  

In [26]:
['doc_sem_similarity',
       'name_sem_similarity', 'name_local_sem_similarity', 'levenshtein_name_stop_phonetic',
       'levenshtein_street_stop_phonetic', 'wratio_name', 'jaro_winkler_name',
       'jaro_winkler_name_stop', 'jaro_winkler_name_local',
       'jaro_winkler_name_local_transliterated', 'jaro_winkler_name_reversed',
       'jw_name_local_ascii_only', 'jw_name_local_nonascii_only',
       'tokenset_name_local', 'tokenset_name_legal',
       'tokenset_name_local_nonascii_only', 'tokenset_name_local_ascii_only',
       'tokenset_name_local_transliterated', 'tokenset_name',
       'tokenset_name_stop', 'tokenset_street_stop', 'wratio_name_stop',
       'wratio_street_stop', 'cosine_name_stop', 'haversine', 'same_phone']

['doc_sem_similarity',
 'name_sem_similarity',
 'name_local_sem_similarity',
 'levenshtein_name_stop_phonetic',
 'levenshtein_street_stop_phonetic',
 'wratio_name',
 'jaro_winkler_name',
 'jaro_winkler_name_stop',
 'jaro_winkler_name_local',
 'jaro_winkler_name_local_transliterated',
 'jaro_winkler_name_reversed',
 'jw_name_local_ascii_only',
 'jw_name_local_nonascii_only',
 'tokenset_name_local',
 'tokenset_name_legal',
 'tokenset_name_local_nonascii_only',
 'tokenset_name_local_ascii_only',
 'tokenset_name_local_transliterated',
 'tokenset_name',
 'tokenset_name_stop',
 'tokenset_street_stop',
 'wratio_name_stop',
 'wratio_street_stop',
 'cosine_name_stop',
 'haversine',
 'same_phone']

In [27]:
features_df_w_semantic_sims.to_gbq("dh-global-sales-data-dev.leadgen_temp.PH_data_with_all_features_v5_e1_exp")

100%|███████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 7681.88it/s]
